In [97]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ======================
# FEATURE ENGINEERING
# ======================
df['Lag1'] = df['Price'].shift(1)
df['Price_Diff'] = df['Price'] - df['Lag1']
df = df.dropna().reset_index(drop=True)

# ======================
# TRAIN TEST SPLIT
# ======================
split = int(len(df) * 0.8)

train_df = df.iloc[:split]
test_df = df.iloc[split:]

X_train = train_df[['Lag1']]
y_train = train_df['Price_Diff']

X_test = test_df[['Lag1']]
y_test = test_df['Price_Diff']

actual_price = test_df['Price']

# ======================
# MODEL
# ======================
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

# ======================
# PREDICTION
# ======================
pred_diff = rf.predict(X_test)
final_pred = X_test['Lag1'].values + pred_diff

# ======================
# EVALUATION
# ======================
mae = mean_absolute_error(actual_price, final_pred)
rmse = np.sqrt(mean_squared_error(actual_price, final_pred))
r2 = r2_score(actual_price, final_pred)

mape = np.mean(np.abs((actual_price - final_pred) / actual_price)) * 100

print("\n=== RANDOM FOREST RESULT===")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.4f}")
print(f"MAPE: {mape:.2f}")

# save
os.makedirs("models", exist_ok=True)
joblib.dump(rf, "models/random_forest.pkl")


=== RANDOM FOREST RESULT===
MAE: 836.32
RMSE: 1094.65
R2: 0.9972
MAPE: 0.99


['models/random_forest.pkl']